# Khóa luận PS-IPPO — Quy trình thực nghiệm đầy đủ trên Google Colab

**Đề tài:** Xây dựng mô hình học tăng cường tối ưu hóa chính sách đặt hàng lại trong quản lý tồn kho đa kho (10 kho × 30 mặt hàng = 300 tác tử, IPPO với chia sẻ tham số, dữ liệu M5 Walmart).

Notebook trình bày **đúng thứ tự các bước** mà nhóm đã thực hiện để tạo ra kết quả trong báo cáo. Người đọc chỉ cần chạy lần lượt từ trên xuống. **Mã nguồn** lấy từ GitHub; **dữ liệu gốc M5** lấy từ thư mục Google Drive `MyDrive/KLTN_data` (file quá lớn để đưa lên GitHub); mọi kết quả lưu vào **My Drive / `KLTN_final`**.

| Bước | Việc | Đầu ra chính | Thời gian (Colab GPU T4, ước lượng) |
|---|---|---|---|
| 1 | Kết nối Google Drive | — | vài giây |
| 2 | Lấy mã nguồn từ GitHub | `KLTN_final/` | 1 phút |
| 3 | Cài thư viện | — | 1–2 phút |
| 4 | Lấy dữ liệu gốc M5 từ Drive và tiền xử lý | `data/processed/*.npy` | khoảng 1 phút |
| 5 | Kiểm tra dữ liệu và ba miền train / val / test | — | vài giây |
| 6 | Kiểm thử mã nguồn (42 test) | — | vài giây |
| 7 | Tinh chỉnh 3 chính sách cổ điển trên miền validation | `results/baseline_params.json` | khoảng 5 phút |
| 8 | Huấn luyện mô hình chính: 3 seed × 4.000 episode | `checkpoints/best_model_main_s*.pth` | khoảng 1–1,5 giờ mỗi seed |
| 9 | Huấn luyện các thí nghiệm loại trừ (ablation): 10 lần × 1.000 episode | `checkpoints/best_model_abl_*.pth` | khoảng 15–20 phút mỗi lần |
| 10 | Đánh giá trên miền test | `results/*` | khoảng 30–40 phút |
| 11 | Xem nhanh kết quả | — | — |
| 12 | (Tuỳ chọn) Đẩy kết quả lên GitHub | — | — |

**Chuẩn bị trước khi chạy:** tải 3 file của cuộc thi [M5 Forecasting – Accuracy](https://www.kaggle.com/competitions/m5-forecasting-accuracy/data) trên Kaggle — `sales_train_evaluation.csv`, `calendar.csv`, `sell_prices.csv` — và upload vào thư mục **`MyDrive/KLTN_data/`** (Bước 4 sẽ tự tạo thư mục này nếu chưa có).

**Lưu ý khi chạy**
- **Runtime → Change runtime type → T4 GPU.** Huấn luyện dùng GPU: phần cập nhật mạng PPO (chiếm khoảng 2/3 thời gian huấn luyện) chạy trên GPU; phần mô phỏng kho (numpy) vẫn chạy trên CPU của máy Colab. Tổng Bước 8–9 khoảng 6–8 giờ trên GPU, so với khoảng 25 giờ nếu chỉ dùng CPU.
- `train.py` **tự dùng GPU khi có** (in `device=cuda` ở đầu log). Bước 3 kiểm tra GPU trước khi chạy.
- Bước 8–9 dài. Mọi lần huấn luyện đều **tự train tiếp từ checkpoint gần nhất** (lưu mỗi khoảng 25 episode). Nếu Colab ngắt, kết nối lại, chạy lại **Bước 1–4** rồi chạy lại đúng cell đang dở.
- Colab miễn phí có **hạn mức GPU theo ngày**. Khi hết hạn mức, có thể chờ hôm sau, hoặc tạm chuyển runtime sang CPU và chạy tiếp: checkpoint lưu trên GPU nạp được trên CPU và ngược lại, nên không mất tiến độ.

## Bước 1. Kết nối Google Drive

Mọi kết quả (checkpoint, log, bảng, hình) được ghi thẳng vào Drive nên không mất khi Colab ngắt phiên.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Bước 2. Lấy mã nguồn từ GitHub

Nếu thư mục đã có từ lần chạy trước, cell chỉ `git pull --ff-only` (lấy code mới), **không bao giờ ghi đè** kết quả đã sinh ra trên Drive.

In [ ]:
import os
REPO_URL    = "https://github.com/MinhLysA/Khoa_luan.git"   # repo private: https://<TOKEN>@github.com/MinhLysA/Khoa_luan.git
PROJECT_DIR = "/content/drive/MyDrive/KLTN_final"            # thu muc tren My Drive chua code + ket qua
CODE_DIR    = f"{PROJECT_DIR}/rl_inventory"

if not os.path.exists(f"{PROJECT_DIR}/.git"):
    !git clone {REPO_URL} "{PROJECT_DIR}"
else:
    !git -C "{PROJECT_DIR}" pull --ff-only || echo "!!! Khong pull duoc (co thay doi cuc bo) - van dung code hien co tren Drive"

%cd {CODE_DIR}
!git -C "{PROJECT_DIR}" log -1 --format="Phien ban code: %h (%ad) %s" --date=short
os.makedirs("logs", exist_ok=True)

## Bước 3. Cài thư viện và kiểm tra GPU

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import torch
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "| PyTorch", torch.__version__, "| CUDA", torch.version.cuda)
    x = torch.randn(16384, 43, device="cuda"); _ = (x @ torch.randn(43, 128, device="cuda")).sum().item()
    print("GPU hoat dong binh thuong -> train.py se dung device=cuda")
else:
    print("!!! KHONG CO GPU. Vao Runtime -> Change runtime type -> T4 GPU roi chay lai tu Buoc 1.")
    print("    (Van chay duoc tren CPU nhung cham hon khoang 3-4 lan.)")

## Bước 4. Lấy dữ liệu gốc M5 từ Google Drive và tiền xử lý

Ba file CSV gốc của M5 quá lớn để đưa lên GitHub (`sales_train_evaluation.csv` khoảng 120 MB, `sell_prices.csv` khoảng 200 MB), nên được đặt trong thư mục Drive **`MyDrive/KLTN_data/`**.

Cell này:
1. Tạo thư mục `MyDrive/KLTN_data/` nếu chưa có, và kiểm tra đủ 3 file.
2. Chạy **tiền xử lý** (`scripts/data_preprocessing.py`) với đúng tham số trong `config.yaml`: chọn 10 cửa hàng, chọn 30 mặt hàng phân tầng theo quy mô nhu cầu (loại mặt hàng có nhu cầu trung bình dưới 0,2 đơn vị/ngày ở phần lớn cửa hàng), trích đặc trưng lịch và giá. Mọi thống kê chỉ tính trên miền huấn luyện (ngày 0–1049).
3. **So sánh** kết quả với bản dữ liệu đã xử lý lưu trong GitHub, để xác nhận tiền xử lý tái lập đúng dữ liệu đã dùng cho báo cáo.

Đầu ra: `data/processed/demand_data.npy` (1.941 ngày × 10 kho × 30 mặt hàng), `calendar_features.npy`, `price_per_pair.npy`, `price_series.npy`, `env_config.json`.

In [ ]:
import os, shutil, subprocess, numpy as np, yaml
DATA_DIR = "/content/drive/MyDrive/KLTN_data"
CAN = ["sales_train_evaluation.csv", "calendar.csv", "sell_prices.csv"]
os.makedirs(DATA_DIR, exist_ok=True)
thieu = [f for f in CAN if not os.path.exists(f"{DATA_DIR}/{f}")]
if thieu:
    raise SystemExit(f"Thieu {thieu} trong {DATA_DIR}.\n"
                     "Tai tu https://www.kaggle.com/competitions/m5-forecasting-accuracy/data, "
                     "upload vao thu muc tren roi chay lai cell nay.")
print("Du lieu goc:", {f: f"{os.path.getsize(f'{DATA_DIR}/{f}') / 1e6:.0f} MB" for f in CAN})

# Giu lai ban tren GitHub de doi chieu sau khi tien xu ly
GIT_COPY = "/content/processed_tu_github"
shutil.rmtree(GIT_COPY, ignore_errors=True)
shutil.copytree("data/processed", GIT_COPY)

p = yaml.safe_load(open("config.yaml", encoding="utf-8"))
THAM_SO = (f"--n_warehouses {p['env']['n_warehouses']} --n_skus {p['env']['n_skus']} "
           f"--min_mean_demand {p['preprocess']['min_mean_demand']} --split_day {p['env']['split_day']}")
!python -u scripts/data_preprocessing.py --m5 --raw_dir "{DATA_DIR}" --output_dir data/processed {THAM_SO} 2>&1 | tee logs/buoc4_tien_xu_ly.log | tail -6

print("\nDoi chieu voi du lieu da xu ly tren GitHub:")
for f in ["demand_data", "calendar_features", "price_per_pair", "price_series"]:
    a, b = np.load(f"{GIT_COPY}/{f}.npy"), np.load(f"data/processed/{f}.npy")
    same = a.shape == b.shape and np.allclose(a, b, equal_nan=True)
    print(f"  {f:<18} {'TRUNG KHOP' if same else '!!! KHAC - kiem tra lai file goc tren Drive'}")

## Bước 5. Kiểm tra dữ liệu và ba miền thời gian

Chuỗi 1.941 ngày được chia thành ba miền **không chồng lấn**:

| Miền | Ngày | Dùng để |
|---|---|---|
| Train | [0, 1050) | huấn luyện PPO; ước lượng nhu cầu trung bình, sức chứa kho |
| Validation | [1050, 1450) | chọn checkpoint IPPO và tinh chỉnh baseline |
| Test | [1450, 1941) | **chỉ** báo cáo kết quả cuối (491 ngày) |

In [ ]:
import json, yaml, numpy as np
cfg = yaml.safe_load(open("config.yaml", encoding="utf-8"))
d   = np.load("data/processed/demand_data.npy")
cal = np.load("data/processed/calendar_features.npy")
meta = json.load(open("data/processed/env_config.json", encoding="utf-8"))
e = cfg["env"]
print("Nhu cau (ngay, kho, mat hang):", d.shape, "| dac trung lich:", cal.shape)
print("Kho:", meta["stores"])
print(f"Train [0,{e['split_day']})  Val [{e['split_day']},{e['val_day']})  Test [{e['val_day']},{d.shape[0]})")
print(f"Nhu cau TB moi cap: {d[:e['split_day']].mean():.2f} don vi/ngay | ty le ngay bang 0: {(d == 0).mean():.1%}")
assert d.shape == (1941, 10, 30), "Du lieu khong dung kich thuoc - kiem tra lai buoc 4"

## Bước 6. Kiểm thử mã nguồn

Phải thấy **`42 passed`**. Nếu có `failed` thì **dừng lại**, không chạy tiếp.

In [ ]:
!python -m pytest tests -q 2>&1 | tail -2

## Bước 7. Tinh chỉnh ba chính sách cổ điển (EOQ, (s,S), Newsvendor)

Tìm kiếm lưới tham số trên **miền validation**, theo **cùng tiêu chí** dùng để chọn checkpoint IPPO: chi phí vận hành thấp nhất trong số các cấu hình đạt fill rate ≥ 85%.
Đầu ra: `results/baseline_params.json` (tham số được chọn) và `results/baseline_tuning.json` (chi tiết).

In [ ]:
!python -u scripts/tune_baselines.py --episodes 3 --tune_mode val 2>&1 | tee logs/buoc7_tune_baselines.log | grep -E "quet|=>"

## Bước 8. Huấn luyện mô hình chính — 3 seed × 4.000 episode

Cấu hình `config.yaml`: phần thưởng cục bộ theo từng cặp, chuẩn hóa theo cặp, phạt mức phục vụ dạng chuẩn hóa. Mỗi seed chạy độc lập; checkpoint được chọn trên miền validation (chi phí thấp nhất trong số checkpoint đạt fill rate ≥ 85%).

Mỗi cell **tự train tiếp** nếu đã chạy dở, và **tự bỏ qua** nếu đã xong. Huấn luyện chạy trên GPU (`device=cuda`).

**Kiểm tra tốc độ trước khi để chạy dài:** cell 8.0 huấn luyện thử 30 episode trên GPU (khoảng 1 phút), in thời gian mỗi episode rồi xóa kết quả thử. Trên T4 nên thấy khoảng 1 giây/episode hoặc ít hơn.

In [ ]:
# 8.0. Chay thu 30 episode tren GPU de do toc do, xong xoa ket qua thu
import time, glob, os, shutil
t = time.time()
!python -u scripts/train.py --episodes 30 --tag gpu_test --device cuda 2>&1 | grep -E "device=|Ep +30|Error|error" | head -5
print(f"=> {(time.time() - t) / 30:.2f} giay/episode (gom ca thoi gian khoi dong)")
for f in glob.glob("checkpoints/*gpu_test*") + glob.glob("results/*gpu_test*"):
    os.remove(f)
for d in glob.glob("runs/*gpu_test*"):
    shutil.rmtree(d, ignore_errors=True)

In [ ]:
!python -u scripts/campaign.py train main_s42 2>&1 | tee -a logs/buoc8_main_s42.log

In [ ]:
!python -u scripts/campaign.py train main_s1 2>&1 | tee -a logs/buoc8_main_s1.log

In [ ]:
!python -u scripts/campaign.py train main_s2 2>&1 | tee -a logs/buoc8_main_s2.log

## Bước 9. Huấn luyện các thí nghiệm loại trừ (ablation) — seed 42 × 1.000 episode

Mỗi thí nghiệm chỉ khác lần chạy tham chiếu `abl_ref` **đúng một thay đổi** (cấu hình trong `configs/`, sinh bằng `configs/make_configs.py`). Mọi lần chạy dùng cùng seed và cùng ngân sách 1.000 episode, nên so sánh được với nhau.

| Nhóm | Lần chạy | Thay đổi | Trả lời |
|---|---|---|---|
| Mốc | `abl_ref` | — (cấu hình chính, 1.000 episode) | mốc so sánh |
| RQ3 | `abl_global` | phần thưởng toàn cục thay cho cục bộ | cục bộ hay toàn cục tốt hơn |
| RQ3 | `abl_q3` | tắt chuẩn hóa phần thưởng theo cặp | vai trò của chuẩn hóa |
| Phần thưởng | `abl_reward_cu` | phạt mức phục vụ kiểu cũ | tác động của việc định cỡ lại hình phạt |
| RQ2 | `holdout` | huấn luyện 24 mặt hàng, đánh giá 6 mặt hàng chưa gặp | tổng quát hóa sang mặt hàng mới |
| Kiến trúc | `abl_trunk` | Actor/Critic dùng chung thân mạng | vì sao phải tách hai mạng |
| Trạng thái | `abl_nocal`, `abl_nowh` | bỏ đặc trưng lịch / bỏ tín hiệu cấp kho | trạng thái có biến dư thừa không |
| Trạng thái | `abl_event` | thêm đặc trưng sự kiện sắp tới | cải thiện mùa lễ |
| Giao thức | `abl_warm` | lịch sử nhu cầu khởi tạo bằng dữ liệu thật | ảnh hưởng của lịch sử rỗng |

In [ ]:
# 8a. Moc so sanh va cac thi nghiem cho RQ3
for tag in ["abl_ref", "abl_global", "abl_q3"]:
    !python -u scripts/campaign.py train {tag} 2>&1 | tee -a logs/buoc9_{tag}.log

In [ ]:
# 8b. Phan thuong va tong quat hoa
for tag in ["abl_reward_cu", "holdout"]:
    !python -u scripts/campaign.py train {tag} 2>&1 | tee -a logs/buoc9_{tag}.log

In [ ]:
# 8c. Kien truc va trang thai
for tag in ["abl_trunk", "abl_nocal", "abl_nowh", "abl_event", "abl_warm"]:
    !python -u scripts/campaign.py train {tag} 2>&1 | tee -a logs/buoc9_{tag}.log

In [ ]:
# Kiem tra: moi dong phai la XONG truoc khi sang Buoc 10
!python scripts/campaign.py status

## Bước 10. Đánh giá trên miền test

**Giao thức:** mỗi lần đánh giá chạy **trọn 491 ngày** của miền test với nhu cầu thật cố định, lặp **30 hạt giống** thời gian giao hàng. Mọi chính sách dùng cùng hạt giống, nên kiểm định là **theo cặp**: chênh lệch trung bình, khoảng tin cậy 95%, paired t-test, Wilcoxon, hệ số d_z.

### 9.1. So sánh chính (RQ1): mô hình chính 3 seed so với 3 chính sách cổ điển

In [ ]:
for s in [42, 1, 2]:
    !python -u scripts/evaluate.py --checkpoint checkpoints/best_model_main_s{s}.pth --episodes 30 --tag main_s{s} 2>&1 | tee logs/buoc10_eval_main_s{s}.log | grep -E "IPPO -|IPPO  "
    !python scripts/plot_learning_curve.py --tag main_s{s}
!python scripts/campaign.py tonghop

### 9.2. Phân tích phụ: so sánh ở cùng mức phục vụ (~ mức IPPO đạt được)

In [ ]:
!python -u scripts/iso_service.py --checkpoint checkpoints/best_model_main_s42.pth --tune_episodes 3 --eval_episodes 30 2>&1 | tee logs/buoc10_iso.log | tail -12

### 9.3. Thực nghiệm B (theo giai đoạn nhu cầu, sốc cầu), hành vi chính sách, nhóm quy mô cầu, độ nhạy chi phí

In [ ]:
CK = "checkpoints/best_model_main_s42.pth"
!python -u scripts/regime_analysis.py      --checkpoint {CK} --tag main 2>&1 | tee logs/buoc10_regime.log   | grep CI95
!python -u scripts/policy_behavior.py      --checkpoint {CK} --tag main 2>&1 | tee logs/buoc10_behavior.log | tail -9
!python -u scripts/analyze_scale_groups.py --checkpoint {CK} --tag main 2>&1 | tee logs/buoc10_scale.log    | tail -3
!python -u scripts/sensitivity.py          --checkpoint {CK} --tag main 2>&1 | tee logs/buoc10_sensitivity.log | tail -6

### 9.4. Đánh giá các thí nghiệm loại trừ (mỗi lần với đúng cấu hình đã dùng khi train)

In [ ]:
ABL = {"abl_ref": "config.yaml",
       "abl_global": "configs/ablation_global_reward.yaml",
       "abl_q3": "configs/ablation_q3_no_reward_norm.yaml",
       "abl_reward_cu": "configs/ablation_reward_demand_scaled.yaml",
       "abl_trunk": "configs/ablation_shared_trunk.yaml",
       "abl_nocal": "configs/ablation_drop_calendar.yaml",
       "abl_nowh": "configs/ablation_drop_warehouse.yaml",
       "abl_event": "configs/ablation_event_lookahead.yaml",
       "abl_warm": "configs/ablation_warm_start.yaml"}
for tag, cfgf in ABL.items():
    !python -u scripts/evaluate.py --config {cfgf} --checkpoint checkpoints/best_model_{tag}.pth --episodes 30 --tag {tag} 2>&1 | tee logs/buoc10_eval_{tag}.log | grep "IPPO  "
# Phan bo muc phuc vu tung cap: phat SLA moi so voi kieu cu
!python scripts/policy_behavior.py --config config.yaml --checkpoint checkpoints/best_model_abl_ref.pth --tag abl_ref
!python scripts/policy_behavior.py --config configs/ablation_reward_demand_scaled.yaml --checkpoint checkpoints/best_model_abl_reward_cu.pth --tag abl_reward_cu
# Duong hoc cua cac ablation ve chong len nhau
!python scripts/plot_learning_curve.py --compare {" ".join(ABL)} --max_episode 1000

### 9.5. Hold-out (RQ2 mở rộng): đánh giá trên 6 mặt hàng chưa gặp khi huấn luyện

In [ ]:
for tag, out in [("holdout", "holdout_unseen"), ("abl_ref", "holdout_seen_ref"), ("main_s42", "holdout_seen_main")]:
    !python -u scripts/evaluate.py --config configs/holdout_eval.yaml --checkpoint checkpoints/best_model_{tag}.pth --episodes 30 --tag {out} 2>&1 | tee logs/buoc10_{out}.log | grep "IPPO  "

## Bước 11. Xem nhanh kết quả

In [ ]:
import json, glob, os
from IPython.display import Image, display

if os.path.exists("results/multiseed_main.json"):
    d = json.load(open("results/multiseed_main.json", encoding="utf-8"))
    print(f"IPPO (3 seed): chi phi TB {d['ippo_cost_mean']:,.0f} +- {d['ippo_cost_sd']:,.0f} | fill TB {d['ippo_fill_mean']:.1%}")
    for ref, g in d["gap_vs_baseline"].items():
        print(f"  so voi {ref:<12}: {g['mean']:+.2f}% (tu {g['min']:+.2f} den {g['max']:+.2f}); "
              f"co y nghia thong ke o {g['n_seeds_significant']}/{g['n_seeds']} seed")
for png in sorted(glob.glob("results/learning_curve_main_s*.png")) + sorted(glob.glob("results/learning_curve_compare_*.png")) + \
           ["results/regime_analysis_main.png", "results/shock_test_main.png", "results/policy_behavior_main.png"]:
    if os.path.exists(png):
        print(png); display(Image(png))

## Bước 12. (Tuỳ chọn) Đẩy kết quả lên GitHub

Kết quả **đã nằm trên My Drive** (`KLTN_final/rl_inventory/results`, `checkpoints`, `logs`). Cell này chỉ cần khi muốn `git pull` kết quả về máy tính. Token GitHub được nhập lúc chạy, không lưu trong notebook.

In [ ]:
from getpass import getpass
TOKEN = getpass("GitHub token (Enter de bo qua): ")
if TOKEN:
    %cd {PROJECT_DIR}
    !git config user.email "dinhyen.dy205@gmail.com"
    !git config user.name "sulinh3625"
    !git add rl_inventory/results rl_inventory/checkpoints/best_model_* rl_inventory/checkpoints/final_model_* rl_inventory/checkpoints/train_state_*
    !git commit -m "Ket qua dot thuc nghiem cuoi tu Colab"
    !git push https://{TOKEN}@github.com/MinhLysA/Khoa_luan.git HEAD:main
    %cd {CODE_DIR}

---
## Phụ lục — chạy nhiều phiên song song (rút ngắn Bước 8–9)

Trên một GPU T4, Bước 8–9 mất khoảng 6–8 giờ và chạy tuần tự là đủ. Phụ lục này dành cho trường hợp có nhiều phiên cùng lúc: **Colab Pro** (nhiều phiên GPU), hoặc khi hết hạn mức GPU và phải chạy trên CPU (khoảng 25 giờ). Colab miễn phí thường chỉ cho **một** phiên GPU mỗi tài khoản. Cách chạy song song 2–3 phiên:

1. File → *Save a copy in Drive*, mở bản sao ở tab mới.
2. Ở mỗi phiên: chạy **Bước 1–6**, rồi chạy **cell dưới đây** thay cho Bước 8–9. Mở các phiên cách nhau khoảng 5 phút.
3. Mỗi phiên tự nhận một lần huấn luyện chưa phiên nào làm (thứ tự: `main_s42` → `abl_ref`, `abl_global`, `abl_q3` → `main_s1`, `main_s2` → các ablation còn lại). Khi phiên bị ngắt, lần chạy dở được phiên khác nhận lại sau 20 phút và train tiếp từ checkpoint.
4. Khi `campaign.py status` báo mọi dòng **XONG**, quay lại **Bước 10** ở một phiên duy nhất.

In [ ]:
import socket, datetime
LOG = f"logs/songsong_{socket.gethostname()}_{datetime.datetime.now():%Y%m%d_%H%M}.log"
print("Log:", LOG)
!python -u scripts/campaign.py train auto 2>&1 | tee -a "{LOG}"

In [ ]:
# Tien do chung cua moi phien (chay bat cu luc nao)
!python scripts/campaign.py status